# Thumbs_Robot: Unitree G1 Three-Digit Hand-Gesture Control via PPO
## 專案導讀手冊與評分指南 (Notebook Handbook & Grading Guide)

本 Notebook 為 CSCN8020 強化學習期末專案的引導操作手冊。您可以直接在此 Notebook 中逐步執行代碼，完成從環境配置、模型審計、環境測試、模型訓練到結果可視化的完整流程。

### 專案大綱與小組簡介 (Project Overview & Team)
- **專案目標**：使用連續動作的 Actor-Critic (PPO 形式) 控制器，控制 Unitree G1 機器人的三指手部形態（包含兩個主手指與一個大拇指，即 3-digit hand），使其在 MuJoCo 模擬環境中能夠做出三種目標手勢的穩定、平滑且協調的控制：Thumbs Up、Open/Stop 以及 Thumbs Down。
- **學術亮點**：實作教授規定的**「數學 MDP $\rightarrow$ 演算法邏輯 $\rightarrow$ 程式變體 $\rightarrow$ 實時日誌」之四位一體精確對齊映射**。
- **小組成員**：Emmanuel • Liggia • Cemil • Chao

### 1. 環境安裝與設置步驟 (Environment Setup)
為確保本專案可在其他機器上完全再現，請依序執行以下單元格。這將會完成升級 pip、安裝依賴套件、拉取官方第三方 Unitree 倉庫，並驗證 Python 環境。

#### 第零步：升級 pip 並安裝 requirements.txt 中的依賴項目使用 WSL 開啟專案資料夾

1. 在 IDE 中按下 Ctrl + Shift + P（開啟命令面板）。
2. 輸入並選擇：WSL: Connect to WSL
3. 點選 Open Folder，選擇你在 WSL 裡的專案路徑(EX:/mnt/C/Final_Project)
4. 確認狀態：看 IDE 左下角的最下緣狀態列，是否變成了綠色或顯示 WSL: Ubuntu
5. Before installing MuJoCo or Unitree software, install the Linux packages required for Python virtual environments, C/C++ compilation, CMake/Ninja builds, OpenGL rendering, GLFW window management, and WSLg graphics.
```bash
sudo apt update && sudo apt install -y \
    python3-venv \
    python3-dev \
    build-essential \
    git \
    cmake \
    ninja-build \
    pkg-config \
    libglfw3 \
    libglfw3-dev \
    libgl1-mesa-dev \
    libegl1-mesa-dev \
    libxinerama-dev \
    libxcursor-dev \
    libxrandr-dev
```

6. Create a project-local virtual environment so this workshop does not interfere with other Python installations.

```bash
cd /mnt/c/Final_Project

python3 -m venv .venv
source .venv/bin/activate
```

#### 第一步：升級 pip 並安裝 requirements.txt 中的依賴項目

In [2]:
!python -m pip install --upgrade pip setuptools wheel
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 792.3 kB/s  0:00:02 eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 26.2
    Uninstalling pip-26.2:
      Successfully uninstalled pip-26.2


####  第二步：拉取官方第三方 Unitree MuJoCo 倉庫（若 external/unitree_mujoco 不存在）

In [4]:
import os
external_dir = os.path.join("external", "unitree_mujoco")
if not os.path.exists(external_dir):
    print("Cloning unitree_mujoco repository...")
    os.makedirs("external", exist_ok=True)
    !git clone https://github.com/unitreerobotics/unitree_mujoco.git {external_dir}
else:
    print(f"'{external_dir}' already exists, skipping clone.")

'external/unitree_mujoco' already exists, skipping clone.


#### 第三步：驗證主要庫是否安裝成功

In [5]:
import sys
print(f"Python Version: {sys.version}")
try:
    import mujoco
    print(f"MuJoCo Version: {mujoco.__version__}")
except ImportError:
    print("Error: MuJoCo is not installed.")
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
except ImportError:
    print("Error: PyTorch is not installed.")
try:
    import gymnasium as gym
    print(f"Gymnasium Version: {gym.__version__}")
except ImportError:
    print("Error: Gymnasium is not installed.")

Python Version: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
MuJoCo Version: 3.10.0
PyTorch Version: 2.13.0+cu130
CUDA Available: True
Gymnasium Version: 1.3.0


### 2. 模型關節審查與手勢角度校準 [Phase 0]
在開始強化學習之前，我們必須先生成 29-DOF 固定底座 G1 模型，並審查其手腕與手指 Actuators，手動校準出三種手勢的目標關節角度向量。
- **執行檔案**：[create_fixed_base_g1.py](./src/create_fixed_base_g1.py) 與 [g1_model_audit.py](./src/g1_rl/g1_model_audit.py)
- **任務**：載入 G1 機器人手部 XML 模型，印出所有關節（Joints）與致動器（Actuators）名稱與 Limits，手動校準並保存以下三種手勢的目標關節角度向量：
  1. **Thumbs Up (讚)**：大拇指伸展、兩指蜷縮、手腕朝上。
  2. **Open/Stop (掌心張開)**：三指完全伸展、掌心朝前。
  3. **Thumbs Down (倒讚)**：大拇指伸展、兩指蜷縮、手腕朝下。

> **三指形態約束 (3-Digit Hand Rule)**：Unitree G1 使用的是 **兩指主手指 + 一指大拇指** 的三指結構，絕不可使用 5 指人類手掌形態或渲染圖作為技術證據。

In [8]:
# 生成固定底座的 G1 模型並執行模型審計
!python src/create_fixed_base_g1.py
!python src/g1_rl/g1_model_audit.py --xml-path assets/g1_fixed_base/scene_29dof_fixed_base.xml --no-viewer

Removed free joint: floating_base_joint
Created fixed-base model: assets/g1_fixed_base/g1_29dof_fixed_base.xml
Copied meshes to: assets/g1_fixed_base/meshes
Created fixed-base scene: assets/g1_fixed_base/scene_29dof_fixed_base.xml

Fixed-base G1 assets created successfully.
Loading Unitree G1 Model from: assets/g1_fixed_base/scene_29dof_fixed_base.xml

--- Joint Audit ---
Total degrees of freedom (nv): 29
Total number of joints (njnt): 29
Joint ID 00 | Name: 'left_hip_pitch_joint' | Type ID: 3 | Limits: [-2.5307  2.8798]
Joint ID 01 | Name: 'left_hip_roll_joint' | Type ID: 3 | Limits: [-0.5236  2.9671]
Joint ID 02 | Name: 'left_hip_yaw_joint' | Type ID: 3 | Limits: [-2.7576  2.7576]
Joint ID 03 | Name: 'left_knee_joint' | Type ID: 3 | Limits: [-0.087267  2.8798  ]
Joint ID 04 | Name: 'left_ankle_pitch_joint' | Type ID: 3 | Limits: [-0.87267  0.5236 ]
Joint ID 05 | Name: 'left_ankle_roll_joint' | Type ID: 3 | Limits: [-0.2618  0.2618]
Joint ID 06 | Name: 'right_hip_pitch_joint' | Type I

### 3. Gymnasium 環境開發與隨機動作測試 [Phase 1]
- **執行檔案**：[g1_hand_env.py](./src/g1_rl/g1_hand_env.py)
- **任務**：
  * 定義連續狀態空間 $s_t$：$s_t = [q_t, \dot{q}_t, q_{\text{target}}(g), q_{\text{target}}(g)-q_t, \text{one\_hot}(g), a_{t-1}]$
  * 定義連續動作空間 $a_t$：控制關節（手腕與手指）的角度增量。
  * 實作複合獎勵函數：
    $$r_t = w_p(e_{t-1} - e_t) - w_h E_{\text{hand}} - w_o E_{\text{orientation}} - w_v \|\dot{q}_t\|^2 - w_a \|a_t\|^2 - w_s \|a_t-a_{t-1}\|_2^2 + b_{\text{hold}} I_{\text{hold}} - c_{\text{time}}$$
  * 實作維持手勢成功判定（Hold 檢測機制，例如當關節姿態誤差與方向誤差小於閾值並維持 15 步以上，判定為成功完成）。

請執行下方的隨機動作測試單元格，驗證 [g1_hand_env.py](./src/g1_rl/g1_hand_env.py) 的 Gymnasium 封裝是否工作正常：

In [9]:
# 測試 Gymnasium 環境是否能正常初始化與執行 step（隨機動作冒煙測試）
import os
import sys
sys.path.append(os.path.abspath("src"))
from g1_rl.g1_hand_env import G1HandEnv

try:
    # 實例化環境，若 assets/g1_hand.xml 還在開發中，這裏主要作為 API 測試
    env = G1HandEnv(xml_path="assets/g1_fixed_base/scene_29dof_fixed_base.xml")
    obs, info = env.reset()
    print("SUCCESS: Environment initialized successfully!")
    print(f"Observation Space Shape: {obs.shape}")
    print(f"Action Space Shape: {env.action_space.shape}")
    print(f"Initial target gesture: {info.get('target_gesture')}")
    
    # 隨機執行 1 個 Step
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, step_info = env.step(action)
    print("SUCCESS: Environment stepped successfully!")
    print(f"Step Reward: {reward}")
    print(f"Next Obs Shape: {next_obs.shape}")
    print(f"Reward info components: {step_info.get('reward_info')}")
except Exception as e:
    print("Environment test failed. (This is normal if xml model or logic is in placeholder status)")
    print("Error details:", e)

SUCCESS: Environment initialized successfully!
Observation Space Shape: (32,)
Action Space Shape: (5,)
Initial target gesture: OPEN_STOP
SUCCESS: Environment stepped successfully!
Step Reward: -0.3049483299255371
Next Obs Shape: (32,)
Reward info components: {'reward_total': np.float32(-0.30494833), 'progress': 0.0, 'pose_error_penalty': -0.0, 'orientation_penalty': -0.0, 'smoothness_penalty': np.float32(-2.0494833), 'smoothness_delta_penalty': 0.0, 'joint_limit_penalty': 0.0, 'hold_bonus': 0.0, 'success_bonus': 0.0, 'time_penalty': -0.1}


### 4. PPO 演算法架構與網路設計 [Phase 2]
- **執行檔案**：[actor_critic_network.py](./src/Thumbs_Robot/actor_critic_network.py)、[rollout_buffer.py](./src/Thumbs_Robot/rollout_buffer.py)、[agent.py](./src/Thumbs_Robot/agent.py)

- **四位一體映射機制 (Math-to-Code-to-Log Mapping)**：
  本專案嚴格對齊教授要求的四位一體映射，下表展示了數學公式、演算法邏輯、程式碼變數與實時日誌的對應關係：

| 數學概念 (MDP Formula) | 演算法邏輯 (Algorithm) | 程式碼變數 (Code Variable) | Console/CSV 日誌欄位 (Log Field) |
| :--- | :--- | :--- | :--- |
| 當前狀態 $s_t$ | 觀測向量 `env._get_obs()` | `obs` / `state` | `state_error_norm`, `gesture` |
| 策略分布 $\pi_\theta(a \mid s)$ | 輸出均值 $\mu$ 與標準差 $\sigma$ | `action_dist`, `mu`, `std` | `actor_mean`, `actor_std` |
| 選擇動作 $a_t$ | 動作採樣與邊界剪裁 | `raw_action`, `clipped_action` | `action_sample`, `action_clipped` |
| 環境轉移 $p(s' \mid s, a)$ | 模擬器推進物理步 | `next_obs`, `reward`, `terminated` | `V(s_t+1)`, `reward_total` |
| 回合累積獎勵 $G_t$ | 複合獎勵函數計算 | `reward_total`, `reward_components` | `reward_total`, `progress`, `pose`, `hold` |
| 狀態價值 $V_\phi(s)$ | Critic 估計預期回報 | `value_t`, `next_value` | `V(s_t)` |
| TD 目標 $y_t$ | 貝爾曼目標值估算 | `td_target` | `td_target` |
| 優勢估計 $\hat{A}_t$ | 計算 GAE 或 TD 誤差 | `advantage` | `advantage` |
| 策略損失 $L_{\text{actor}}$ | PPO 剪裁策略目標函數 | `actor_loss` | `actor_loss` |
| 價值損失 $L_{\text{critic}}$ | 均方時序差分誤差 | `critic_loss` | `critic_loss` |

- **組件職責**：
  1. **Actor Network**：接收 $s_t$，預測連續動作分布的均值與標準差，使用高斯分布進行採樣。
  2. **Critic Network**：接收 $s_t$，預測期望的狀態價值 $V(s_t)$。
  3. **Rollout Buffer**：儲存 on-policy 軌跡數據（States, Actions, Log_probs, Rewards, Values, Terminals），並計算 GAE 優勢。

### 5. PPO 演算法元件驗證 (PPO Components Verification) [Phase 3]
- **執行檔案**：[smoke_test.py](./src/Thumbs_Robot/smoke_test.py)
- **任務**：
  完善 PyTorch 網路前向傳播（輸出高斯分布均值與標準差，以及狀態價值 $V$）、GAE 優勢估計計算與 PPO Clipped updates。
  執行下方的冒煙測試 (Smoke Test) 腳本，驗證神經網路維度、緩衝區採樣以及優化更新步驟是否完全正常，且無 NaN/Inf 產生。

In [14]:
# 執行冒煙測試，驗證 PPO 元件與網路架構正確性
!python src/Thumbs_Robot/smoke_test.py

Starting PPO Module Smoke Test...

[Step 1] Initializing G1HandEnv in headless mode...
   Environment loaded successfully.
   State space dimension: 32
   Action space dimension: 5

[Step 2] Initializing PPOAgent with test hyperparameters...
   PPOAgent initialized on device: cuda

[Step 3] Running episode to collect 5 steps of transitions...
   Step 01 | Reward: -0.3314 | Done: False | Target: THUMBS_UP
   Step 02 | Reward: -0.4014 | Done: False | Target: THUMBS_UP
   Step 03 | Reward: -0.4795 | Done: False | Target: THUMBS_UP
   Step 04 | Reward: -0.2221 | Done: False | Target: THUMBS_UP
   Step 05 | Reward: -0.2529 | Done: False | Target: THUMBS_UP
   Rollout buffer filled. Current pointer: 5
   Real episode step collection passed.

[Step 4] Computing GAE advantages and executing PPO agent update...
   Calculated advantages shape: torch.Size([5])
/mnt/l/Reinforcement Learning Programming/Final_Project/src/Thumbs_Robot/agent.py:92: UserWarning: Using a target size (torch.Size([1])) t

### 6. 四位一體映射日誌 (Math-to-Code-to-Log Mapping) [Phase 4]
- **任務**：實作步驟與優化更新之 CSV 日誌輸出規格，在 Console 實時印出對齊公式，並驗證「數學 $\rightarrow$ 演算法 $\rightarrow$ 代碼 $\rightarrow$ 日誌」之四位一體映射完全對齊。
- **驗證方式**：運行冒煙更新測試，觀測主控制台輸出的 Update 級別表格日誌（包含 `Loss_A`、`Loss_C`、`Entropy` 等演算法指標與手勢完成度摘要），並檢查 `results/ppo_config_a/` 目錄下靜默寫入的 `episode_log.csv`（包含 Episode 級別的 `Reward`、`Final Pose Error`、`Hold Duration`、`Safety Violation` 等細節日誌），驗證四位一體映射完全對齊。


In [18]:
# 執行冒煙更新測試，檢查 Console 與日誌中的四位一體映射輸出是否符合規定
!python src/Thumbs_Robot/train_thumbs.py --smoke-test

Starting PPO Training: Unitree G1 Hand-Gesture Control
Device: cuda | State Dim: 32 | Action Dim: 5
Random Seed: 666
Hyperparameters:
 - Learning Rate (lr): 0.0003
 - Discount Factor (gamma): 0.99
 - GAE Lambda: 0.95
 - PPO Clip Epsilon: 0.2
 - Batch Size: 2
 - Rollout Length: 10
 - Epochs per Rollout: 2
 - Max Total Steps: 20

Update | Steps    | Ep_Done | Mean_Rwd | Succ_% | Loss_A   | Loss_C   | Entropy | Time   | Last_Gesture
------------------------------------------------------------------------------------------------------
1      | 10       | 0       | N/A      | 0.0%   | -0.0415  | 1.7243   | 7.096   | 1.1  s | N/A         
2      | 20       | 0       | N/A      | 0.0%   | -0.0171  | 2.0875   | 7.098   | 1.1  s | N/A         
------------------------------------------------------------------------------------------------------
Training completed in 1.2 seconds. Saved final model to: models/best_thumbs_robot.pt
Final rolling success rate (last 50 episodes): 0.00%
Mean cumulativ

In [19]:
# 觀看靜默寫入的 episode_log.csv 詳細回合數據
import os
import pandas as pd

episode_log_path = "results/ppo_config_a/episode_log.csv"
if os.path.exists(episode_log_path):
    df = pd.read_csv(episode_log_path)
    print(f"SUCCESS: 讀取到 {len(df)} 筆 Episode 詳細數據。最新 10 筆資料如下：")
    display(df.tail(10))
else:
    print(f"Error: 尚未在 {episode_log_path} 找到日誌，請先執行上方單元格完成訓練。")


SUCCESS: 讀取到 0 筆 Episode 詳細數據。最新 10 筆資料如下：


,Episode,Gesture,Reward,Success,Steps,Final Pose Error,Hold Duration,Safety Violation


### 7. 目標條件多手勢 Headless 訓練 (Target-Conditioned Headless Training) [Phase 5]
- **執行檔案**：[train_thumbs.py](./src/Thumbs_Robot/train_thumbs.py)
- **任務**：正式啟動 Unified Target-Conditioned Policy 的訓練，訓練一個 Actor-Critic (PPO) 模型同時學會 Thumbs Up、Open/Stop 與 Thumbs Down 三種手勢。
- **說明**：訓練日誌與模型權重將會定期寫入 `results/` 與 `models/` 目錄。

In [ ]:
# 啟動正式的連續動作 PPO 多手勢控制訓練 (預設訓練 100,000 步)
# 訓練日誌將寫入 results/ 目錄下
!python src/Thumbs_Robot/train_thumbs.py

### 8. 一鍵評估與 3D 渲染視覺化 [Phase 6]
- **執行檔案**：[evaluate_thumbs.py](./src/Thumbs_Robot/evaluate_thumbs.py) 與 [render_thumbs.py](./src/Thumbs_Robot/render_thumbs.py)
- **任務**：載入最佳檢查點權重，在評估測試中進行 deterministic 評估；並啟動 MuJoCo 3D 視覺化渲染器，展示動態手勢控制。

In [ ]:
# 執行策略評估與成功率統計
!python src/Thumbs_Robot/evaluate_thumbs.py

### 9. 數據可視化與簡報準備 (Visualization & Delivery) [Phase 7]
- **執行檔案**：[plot_results.py](./src/Thumbs_Robot/plot_results.py)
- **任務**：繪製 Return、成功率、Losses 及 Entropy 隨時間的收斂曲線。
請執行下方單元格，繪製訓練曲線並直接在 Notebook 中顯示結果。

In [ ]:
# 繪製訓練收斂圖表
!python src/Thumbs_Robot/plot_results.py

# 在 Notebook 中直接顯示生成的訓練圖表
import os
from IPython.display import Image, display
plot_path = "results/training_curves.png"  # 請根據 plot_results.py 的實際輸出圖片路徑修改
if os.path.exists(plot_path):
    display(Image(filename=plot_path))
else:
    print("未找到訓練曲線圖，請確認訓練已完成且 plot_results.py 已成功運行並生成該圖片。")